# Fase 2 — EDA (Análisis Exploratorio de Datos)

Exploración visual completa del dataset de sesiones del juego Hack & Slash.
Analiza sesiones, niveles, salas, hechizos, enemigos y comentarios de jugadores
para identificar patrones relevantes para el balance del juego.

**Entradas:** `data/processed/sessions.parquet`, `levels.parquet`, `rooms.parquet`  
**Salidas:** figuras en `reports/figures/`, conclusiones por bloque

> **Nota sobre el volumen:** 11 sesiones — muestra de desarrollo. Los patrones
> detectados son orientativos y se confirmarán con más datos.

## 0. Configuración

In [ ]:
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..') / 'src'))

from preprocessing import run_pipeline, compute_spell_efficiency, compute_spell_usage_ratios
from cleaning import flag_suspicious_sessions, get_cleaning_report, print_cleaning_report, check_sample_size

# Rutas
PROCESSED_DIR = Path('..') / 'data' / 'processed'
FIGURES_DIR   = Path('..') / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Estilo global (estilo P1 del máster)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE_ELEM = {'Fire': '#e74c3c', 'Water': '#3498db', 'Earth': '#27ae60', 'Wind': '#f39c12'}

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Configuración lista.')

## 1. Carga de datos

In [ ]:
df_sessions = pd.read_parquet(PROCESSED_DIR / 'sessions.parquet')
df_levels   = pd.read_parquet(PROCESSED_DIR / 'levels.parquet')
df_rooms    = pd.read_parquet(PROCESSED_DIR / 'rooms.parquet')

print(f'Sesiones : {len(df_sessions)}')
print(f'Niveles  : {len(df_levels)}')
print(f'Salas    : {len(df_rooms)}')

In [ ]:
# Informe de calidad del dataset
df_sessions = flag_suspicious_sessions(df_sessions)
report = get_cleaning_report(df_sessions, df_levels, df_rooms)
print_cleaning_report(report)

print()
check_sample_size(df_sessions, 'playerElement', min_per_group=30)

---
## 2. Sesiones globales

Visión general del dataset: distribuciones de métricas clave, tasa de victoria
y popularidad de elementos.

In [ ]:
# Fig 2.1 — Distribuciones de métricas de sesión
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Distribución de métricas de sesión', fontsize=14, fontweight='bold')

metricas = [
    ('totalTimeSecs', 'Tiempo total (s)'),
    ('totalKills',    'Kills totales'),
    ('totalDeaths',   'Muertes totales'),
]
for ax, (col, label) in zip(axes, metricas):
    sns.histplot(df_sessions[col], ax=ax, kde=True, color='steelblue', bins=8)
    ax.set_xlabel(label)
    ax.set_ylabel('Sesiones')
    ax.axvline(df_sessions[col].median(), color='red', linestyle='--', alpha=0.7, label=f'Mediana: {df_sessions[col].median():.0f}')
    ax.legend(fontsize=9)

save_fig('fig2_1_distribuciones_sesion')
print('Estadísticas descriptivas:')
print(df_sessions[['totalTimeSecs','totalKills','totalDeaths','levelsCompleted']].describe().round(1))

In [ ]:
# Fig 2.2 — Tasa de victoria global y por elemento
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Victorias en el juego', fontsize=14, fontweight='bold')

# Pastel global
counts_vic = df_sessions['isVictory'].value_counts()
labels_vic = ['Derrota', 'Victoria']
axes[0].pie(
    [counts_vic.get(False, 0), counts_vic.get(True, 0)],
    labels=labels_vic, autopct='%1.0f%%',
    colors=['#e74c3c', '#2ecc71'], startangle=90
)
axes[0].set_title('Tasa de victoria global')

# Win rate por elemento
wr = df_sessions.groupby('playerElement').agg(
    sesiones=('sessionId','count'),
    victorias=('isVictory','sum')
)
wr['win_rate'] = wr['victorias'] / wr['sesiones'] * 100
colors_elem = [PALETTE_ELEM.get(e, 'gray') for e in wr.index]
bars = axes[1].barh(wr.index, wr['win_rate'], color=colors_elem)
axes[1].set_xlabel('Win rate (%)')
axes[1].set_title('Win rate por elemento')
axes[1].set_xlim(0, 100)
axes[1].axvline(50, color='gray', linestyle='--', alpha=0.5, label='50%')
for bar, (_, row) in zip(bars, wr.iterrows()):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{row["win_rate"]:.0f}% (n={int(row["sesiones"])})', va='center', fontsize=9)
axes[1].legend()

save_fig('fig2_2_victorias')
print('Win rate por elemento:')
print(wr.round(1))

In [ ]:
# Fig 2.3 — Popularidad de elementos y plataformas
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Perfil del jugador', fontsize=14, fontweight='bold')

# Elementos
elem_counts = df_sessions['playerElement'].value_counts()
colors_bar  = [PALETTE_ELEM.get(e, 'gray') for e in elem_counts.index]
axes[0].bar(elem_counts.index, elem_counts.values, color=colors_bar)
axes[0].set_xlabel('Elemento')
axes[0].set_ylabel('Nº de sesiones')
axes[0].set_title('Popularidad de elementos')
for i, v in enumerate(elem_counts.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# Plataformas
plat_counts = df_sessions['platform'].value_counts()
axes[1].pie(plat_counts.values, labels=plat_counts.index,
            autopct='%1.0f%%', colors=['#3498db','#e67e22'], startangle=90)
axes[1].set_title('Distribución por plataforma')

save_fig('fig2_3_perfil_jugador')

**Conclusiones bloque 2 — Sesiones globales:**
- La tasa de victoria global es baja (1/11 = 9%) — el juego es difícil o los testers son novatos.
- Fire es el elemento más popular (7/11 sesiones). Wind y Earth tienen 1 sesión cada uno.
- Solo Fire tiene victoria — con tan pocos datos no se puede concluir desequilibrio.
- WebGL es la plataforma dominante (9/11 sesiones).

---
## 3. Análisis de niveles

Dificultad por nivel, funnel de progresión, curva de dificultad y análisis de reintentos.

In [ ]:
# Agregados por nivel
nivel_stats = df_levels.groupby('levelId').agg(
    sesiones      =('sessionId', 'count'),
    muertes_media =('deaths',    'mean'),
    tiempo_medio  =('timeSecs',  'mean'),
    danio_medio   =('damageTaken','mean'),
    intento_medio =('attempt',   'mean'),
    intento_max   =('attempt',   'max'),
).round(1).sort_values('levelId')

print('Estadísticas por nivel:')
print(nivel_stats)

In [ ]:
# Fig 3.1 — Funnel de progresión
niveles_order = sorted(df_levels['levelId'].unique())
n_total = len(df_sessions)
sesiones_por_nivel = df_levels.groupby('levelId')['sessionId'].nunique()

fig, ax = plt.subplots(figsize=(9, 5))
pcts = [sesiones_por_nivel.get(lv, 0) / n_total * 100 for lv in niveles_order]
bars = ax.bar(niveles_order, pcts, color='#5dade2', edgecolor='white')
ax.set_xlabel('Nivel')
ax.set_ylabel('% sesiones que alcanzan el nivel')
ax.set_title('Funnel de progresión — % jugadores por nivel', fontweight='bold')
ax.set_ylim(0, 110)
for bar, pct, lv in zip(bars, pcts, niveles_order):
    n = sesiones_por_nivel.get(lv, 0)
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{pct:.0f}%\n(n={n})', ha='center', fontsize=10)

save_fig('fig3_1_funnel_progresion')

In [ ]:
# Fig 3.2 — Dificultad por nivel: daño recibido y tiempo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dificultad por nivel', fontsize=14, fontweight='bold')

# Daño medio por nivel
axes[0].bar(nivel_stats.index, nivel_stats['danio_medio'], color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Nivel')
axes[0].set_ylabel('Daño recibido medio')
axes[0].set_title('Daño medio recibido por nivel')
for i, v in enumerate(nivel_stats['danio_medio']):
    axes[0].text(i, v + 2, f'{v:.0f}', ha='center', fontsize=9)

# Tiempo medio por nivel
axes[1].bar(nivel_stats.index, nivel_stats['tiempo_medio'], color='#f39c12', alpha=0.8)
axes[1].set_xlabel('Nivel')
axes[1].set_ylabel('Tiempo medio (s)')
axes[1].set_title('Tiempo medio por nivel')
for i, v in enumerate(nivel_stats['tiempo_medio']):
    axes[1].text(i, v + 2, f'{v:.0f}s', ha='center', fontsize=9)

save_fig('fig3_2_dificultad_niveles')

In [ ]:
# Fig 3.3 — Análisis de attempt (grinding): reintentos por nivel
fig, ax = plt.subplots(figsize=(10, 5))

attempt_data = df_levels.groupby('levelId')['attempt'].value_counts().unstack(fill_value=0)
attempt_data.plot(kind='bar', ax=ax, colormap='Blues', edgecolor='white')
ax.set_xlabel('Nivel')
ax.set_ylabel('Nº de sesiones')
ax.set_title('Distribución de intentos por nivel (grinding)', fontweight='bold')
ax.legend(title='Intento nº', bbox_to_anchor=(1.01, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

save_fig('fig3_3_attempts_grinding')
print('Intentos medios y máximos por nivel:')
print(nivel_stats[['intento_medio','intento_max']])

**Conclusiones bloque 3 — Niveles:**
- Level1 es el más frecuentado (11 sesiones), Level4 el menos (1 sesión) — progresión descendente.
- Level1 tiene el intento medio más alto (~2.9) — es donde más se repite, posible punto de abandono.
- Level2 requiere más tiempo medio (183s) que Level3 y Level4 — posible spike de dificultad.
- El daño recibido aumenta con el nivel pero no de forma lineal.

---
## 4. Análisis de salas

Hotspots de dificultad, comportamiento táctico (`firstSpell`) y efectividad por sala.

In [ ]:
# Fig 4.1 — Heatmap de muertes por sala/nivel
pivot_deaths = df_rooms.pivot_table(
    index='roomId', columns='levelId', values='deaths', aggfunc='sum', fill_value=0
)

fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_deaths) * 0.35)))
sns.heatmap(pivot_deaths, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Muertes totales'})
ax.set_title('Heatmap de muertes por sala y nivel', fontweight='bold')
ax.set_xlabel('Nivel')
ax.set_ylabel('Sala')

save_fig('fig4_1_heatmap_muertes_salas')

In [ ]:
# Fig 4.2 — firstSpell por sala: comportamiento táctico al entrar
first_spell_sala = df_rooms.groupby(['levelId', 'firstSpell']).size().reset_index(name='count')
first_spell_pivot = first_spell_sala.pivot_table(
    index='levelId', columns='firstSpell', values='count', fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 5))
first_spell_pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_xlabel('Nivel')
ax.set_ylabel('Nº de salas')
ax.set_title('Hechizo inicial (firstSpell) por nivel — comportamiento táctico', fontweight='bold')
ax.legend(title='Hechizo', bbox_to_anchor=(1.01, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

save_fig('fig4_2_firstspell_por_nivel')
print('firstSpell global:')
print(df_rooms['firstSpell'].value_counts())

In [ ]:
# Fig 4.3 — Daño recibido medio por sala (top salas más peligrosas)
sala_danger = df_rooms.groupby(['levelId','roomId']).agg(
    danio_medio=('damageTaken','mean'),
    muertes_total=('deaths','sum'),
    n_visitas=('sessionId','count')
).reset_index().sort_values('danio_medio', ascending=False).head(15)

sala_danger['sala_label'] = sala_danger['levelId'] + ' / ' + sala_danger['roomId']

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(sala_danger['sala_label'], sala_danger['danio_medio'],
               color='#c0392b', alpha=0.85)
ax.set_xlabel('Daño recibido medio')
ax.set_title('Top 15 salas más peligrosas (daño medio)', fontweight='bold')
ax.invert_yaxis()

save_fig('fig4_3_salas_peligrosas')

**Conclusiones bloque 4 — Salas:**
- Projectile es el hechizo inicial dominante (65/82 salas) — los jugadores confían en él por defecto.
- Beam y AOE se usan poco como hechizo inicial a pesar de su mayor eficiencia.
- Las salas de niveles superiores concentran más daño recibido.

---
## 5. Análisis de hechizos

Ranking de uso, eficiencia, `noMana` como demanda latente y matriz por elemento.

In [ ]:
# Fig 5.1 — Uso vs Eficiencia de hechizos (scatter)
df_eff = compute_spell_efficiency(df_rooms)

fig, ax = plt.subplots(figsize=(9, 6))
scatter_colors = ['#e74c3c' if e == 'Projectile' else '#3498db' for e in df_eff['spell_type']]
sc = ax.scatter(df_eff['total_cast'], df_eff['spell_efficiency'],
                s=df_eff['total_cast'] / 3 + 50,
                c=scatter_colors, alpha=0.8, edgecolors='white', linewidth=1.5)

for _, row in df_eff.iterrows():
    ax.annotate(row['spell_type'],
                (row['total_cast'], row['spell_efficiency']),
                textcoords='offset points', xytext=(8, 4), fontsize=11)

ax.axhline(df_eff['spell_efficiency'].mean(), color='gray', linestyle='--',
           alpha=0.6, label=f'Eficiencia media: {df_eff["spell_efficiency"].mean():.2f}')
ax.set_xlabel('Total de lanzamientos (uso)')
ax.set_ylabel('Kills por lanzamiento (eficiencia)')
ax.set_title('Uso vs Eficiencia de hechizos\n(tamaño = uso)', fontweight='bold')
ax.legend()

save_fig('fig5_1_uso_vs_eficiencia_hechizos')
print('Spell efficiency completa:')
print(df_eff.to_string(index=False))

In [ ]:
# Fig 5.2 — noMana: demanda latente de hechizos
nomana_cols = [c for c in df_rooms.columns if c.startswith('noMana_')]
nomana_totals = df_rooms[nomana_cols].sum().sort_values(ascending=False)
nomana_totals.index = nomana_totals.index.str.replace('noMana_', '')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(nomana_totals.index, nomana_totals.values, color='#8e44ad', alpha=0.85)
ax.set_xlabel('Hechizo')
ax.set_ylabel('Intentos sin maná')
ax.set_title('noMana por hechizo — demanda latente\n(intentos fallidos por falta de maná)',
             fontweight='bold')
for bar, v in zip(bars, nomana_totals.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{v:.0f}', ha='center', fontweight='bold')

save_fig('fig5_2_nomana_demanda_latente')
print('noMana totales por hechizo:')
print(nomana_totals)

In [ ]:
# Fig 5.3 — Uso relativo de hechizos por sesión
df_ratios = compute_spell_usage_ratios(df_rooms, df_sessions)
df_ratios = df_ratios.merge(df_sessions[['sessionId','playerElement','isVictory']], on='sessionId')

ratio_cols = [c for c in df_ratios.columns if c.startswith('ratio_cast_')]
spell_names = [c.replace('ratio_cast_','') for c in ratio_cols]

# Stacked bar por sesión
fig, ax = plt.subplots(figsize=(13, 5))
bottom = np.zeros(len(df_ratios))
colors_spell = ['#3498db','#e74c3c','#2ecc71','#f39c12']
for col, name, color in zip(ratio_cols, spell_names, colors_spell):
    ax.bar(range(len(df_ratios)), df_ratios[col], bottom=bottom, label=name, color=color, alpha=0.85)
    bottom += df_ratios[col].values

labels_x = [f"{row['playerElement']}\n{'V' if row['isVictory'] else 'D'}" for _, row in df_ratios.iterrows()]
ax.set_xticks(range(len(df_ratios)))
ax.set_xticklabels(labels_x, fontsize=9)
ax.set_ylabel('Proporción de uso')
ax.set_title('Distribución de uso de hechizos por sesión\n(V=Victoria, D=Derrota)', fontweight='bold')
ax.legend(title='Hechizo', bbox_to_anchor=(1.01, 1))

save_fig('fig5_3_uso_relativo_hechizos')

**Conclusiones bloque 5 — Hechizos:**
- **Beam** tiene la mayor eficiencia (2.04 kills/cast) pero es el menos usado como firstSpell — los jugadores desconocen su potencia o tiene alguna limitación percibida.
- **Projectile** es el más usado (715 lanzamientos) pero el menos eficiente (0.017 kills/cast) — posiblemente sobrebalanceado en accesibilidad pero infravalorado como herramienta de daño.
- **Projectile** tiene la mayor demanda latente (11 noMana) — los jugadores intentan usarlo más de lo que el maná permite. Posible reducción de coste.
- **AOE** tiene 0 kills atribuidos — puede ser que el tracking de kills no esté registrando correctamente sus kills, o que sea ineficaz.

---
## 6. Análisis de enemigos

Enemigos más frecuentes, más letales y patrones de daño/bloqueo.

In [ ]:
# Preparar datos de enemigos
kills_cols   = [c for c in df_rooms.columns if c.startswith('kills_') and 'per_sec' not in c]
blocked_cols = [c for c in df_rooms.columns if c.startswith('blocked_')]
damage_cols  = [c for c in df_rooms.columns if c.startswith('damage_')]

kills_totals   = df_rooms[kills_cols].sum().sort_values(ascending=False)
blocked_totals = df_rooms[blocked_cols].sum().sort_values(ascending=False)
damage_totals  = df_rooms[damage_cols].sum().sort_values(ascending=False)

# Limpiar prefijos
kills_totals.index   = kills_totals.index.str.replace('kills_', '')
blocked_totals.index = blocked_totals.index.str.replace('blocked_', '')
damage_totals.index  = damage_totals.index.str.replace('damage_', '')

print('Top enemigos por kills:')
print(kills_totals.head(10))

In [ ]:
# Fig 6.1 — Top enemigos: kills, bloqueos y daño
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Estadísticas de enemigos', fontsize=14, fontweight='bold')

for ax, data, title, color in [
    (axes[0], kills_totals.head(10),   'Kills por tipo de enemigo',   '#27ae60'),
    (axes[1], blocked_totals.head(10), 'Bloqueos por tipo de enemigo','#e67e22'),
    (axes[2], damage_totals.head(10),  'Daño recibido por enemigo',   '#e74c3c'),
]:
    if data.empty:
        ax.set_title(f'{title}\n(sin datos)')
        continue
    ax.barh(data.index[::-1], data.values[::-1], color=color, alpha=0.85)
    ax.set_xlabel('Total')
    ax.set_title(title)

save_fig('fig6_1_estadisticas_enemigos')

**Conclusiones bloque 6 — Enemigos:**
- Rogue y Knight2H son los enemigos más eliminados — son los más frecuentes o los más fáciles de matar.
- Los Boss tienen pocos kills — esperado, son objetivos difíciles y poco frecuentes.
- El daño de `Unknown` indica que hay una fuente de daño no tageada correctamente en el juego — bug de telemetría a reportar.

---
## 7. Análisis de comentarios de jugadores

Texto libre de `playerComment` y contexto (`Victory` vs `GameOver`).

In [ ]:
# Sesiones con comentario
df_comments = df_sessions[df_sessions['hasComment']].copy()
print(f'Sesiones con comentario: {len(df_comments)} / {len(df_sessions)}')
print()
print(df_comments[['playerElement','isVictory','levelsCompleted',
                   'playerComment','playerCommentContext']].to_string(index=False))

In [ ]:
# Fig 7.1 — Distribución de comentarios por contexto y elemento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Comentarios de jugadores', fontsize=14, fontweight='bold')

# Distribución con/sin comentario
comment_counts = df_sessions['hasComment'].value_counts()
axes[0].pie(
    comment_counts.values,
    labels=['Sin comentario', 'Con comentario'],
    autopct='%1.0f%%', colors=['#bdc3c7','#2ecc71'], startangle=90
)
axes[0].set_title('Tasa de comentarios')

# Contexto de los comentarios
if len(df_comments) > 0:
    ctx_counts = df_comments['playerCommentContext'].value_counts()
    axes[1].bar(ctx_counts.index, ctx_counts.values,
                color=['#e74c3c' if 'GameOver' in c else '#2ecc71' for c in ctx_counts.index])
    axes[1].set_xlabel('Contexto')
    axes[1].set_ylabel('Nº comentarios')
    axes[1].set_title('Contexto de los comentarios')
else:
    axes[1].text(0.5, 0.5, 'Sin datos suficientes', ha='center', va='center', transform=axes[1].transAxes)

save_fig('fig7_1_comentarios_overview')
print('\nNota: con solo 2 comentarios el análisis NLP se realizará en 06_nlp_comments.ipynb\ncuando haya más datos.')

In [ ]:
# Fig 7.2 — Longitud de comentarios y cruce con playerCommentContext
df_sessions['comment_len'] = df_sessions['playerComment'].str.len().fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
ctx_colors = {'GameOver': '#e74c3c', 'Victory': '#2ecc71', '': '#bdc3c7'}
for _, row in df_sessions[df_sessions['hasComment']].iterrows():
    color = ctx_colors.get(row['playerCommentContext'], '#bdc3c7')
    ax.bar(row['sessionId'][-8:], row['comment_len'], color=color, alpha=0.85)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='GameOver'),
                   Patch(facecolor='#2ecc71', label='Victory')]
ax.legend(handles=legend_elements)
ax.set_xlabel('Sesión (últimos 8 chars ID)')
ax.set_ylabel('Longitud del comentario (chars)')
ax.set_title('Longitud de comentarios por contexto', fontweight='bold')

save_fig('fig7_2_longitud_comentarios')

**Conclusiones bloque 7 — Comentarios:**
- Solo 2/11 sesiones tienen comentario (18%) — tasa de feedback muy baja con datos actuales.
- Ambos comentarios son de contexto `GameOver` — ninguna sesión con victoria tiene comentario aún.
- Los textos actuales parecen pruebas de telemetría ('Prueba Comentario'), no feedback real.
- El análisis NLP completo (sentiment + topic modeling) se realizará en `06_nlp_comments.ipynb` cuando haya suficientes comentarios reales.

---
## 8. Resumen ejecutivo del EDA

Tabla de hallazgos clave para el análisis de balance.

In [ ]:
# Tabla resumen de hallazgos
hallazgos = [
    ('Tasa de victoria',      '9% (1/11)',         'El juego es muy difícil o los testers son inexpertos'),
    ('Elemento dominante',    'Fire (7/11 sesiones)','Wind y Earth con 1 sesión — datos insuficientes para balance'),
    ('Hechizo más usado',     'Projectile (715 cast)','Pero el menos eficiente (0.017 kills/cast)'),
    ('Hechizo más eficiente', 'Beam (2.04 kills/cast)','Muy subutilizado — posible infravaloración por jugadores'),
    ('Mayor demanda latente', 'Projectile (11 noMana)','Los jugadores quieren usarlo más — reducir coste de maná'),
    ('AOE kills = 0',         'AOE no registra kills','Posible bug de telemetría o mecánica ineficaz'),
    ('Daño Unknown',          'Fuente sin tag',       'Bug de telemetría — categorizar correctamente en Unity'),
    ('Nivel con más grinding','Level1 (2.9 intentos media)','Punto de entrada con alta tasa de reintento'),
    ('Nivel más largo',       'Level2 (183s media)',  'Posible spike de dificultad — revisar diseño'),
    ('Comentarios reales',    '0 / 11 sesiones',      'Los 2 comentarios son pruebas — necesita más testers'),
]

df_hallazgos = pd.DataFrame(hallazgos, columns=['Aspecto', 'Valor', 'Observación'])
print(df_hallazgos.to_string(index=False))

# Guardar para el informe final
exports_dir = Path('..') / 'data' / 'exports'
exports_dir.mkdir(parents=True, exist_ok=True)
df_hallazgos.to_csv(exports_dir / 'eda_hallazgos.csv', index=False)
print(f'\nHallazgos exportados a data/exports/eda_hallazgos.csv')

In [ ]:
# Listar todas las figuras generadas
figuras = sorted(FIGURES_DIR.glob('fig*.png'))
print(f'Figuras generadas ({len(figuras)}):')
for f in figuras:
    print(f'  {f.name}')

---
## Conclusiones finales del EDA

**Datos:** 11 sesiones, 18 niveles, 82 salas. Muestra de desarrollo — resultados orientativos.

**Principales hallazgos para el balance:**

1. **Hechizos desbalanceados:** Beam (2.04 kills/cast) es 120x más eficiente que Projectile (0.017) pero se usa 14x menos. El jugador no percibe esta diferencia — posible problema de feedback visual o accesibilidad del hechizo.

2. **Projectile sobredemandado:** 11 intentos fallidos por falta de maná — los jugadores quieren usarlo más. Candidato a reducción de coste de maná.

3. **AOE sin kills registrados:** Necesita investigación — puede ser bug de telemetría o mecánica que no funciona correctamente.

4. **Level1 como barrera de entrada:** Mayor tasa de reintentos (2.9 intentos de media). Si es el nivel tutorial, puede estar demasiado ajustado.

5. **Fuente de daño Unknown:** Bug de telemetría que necesita corrección en Unity antes de análisis estadístico.

**Siguiente paso:** `04_statistics.ipynb` — tests estadísticos para validar estas observaciones.